**Author(s):** Oliver Baker, Martin Ross  
**Affiliation(s):** Earth and Environmental Sciences, University of Waterloo  
**Date:** 2026-05-06 

---

## Overview
Code used for Gaussian Mixture Modelling in submitted manuscript to Quaternary Science Reviews titled 
"Improving regional paleo-ice sheet reconstructions with till provenance analysis: A new approach applied to northern Quebec and Labrador" 

## Data
Will be updated after manuscript acceptance

## Note on AI use
ChapGPT (GPT-5) was used to improve and clean the code, fix some issues, and for annotations

In [ ]:
# -------------------------
# BIC Search of Parameters
#--------------------------

import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn import metrics
import matplotlib.pyplot as plt
import itertools #to cycle through marker styles
import numpy as np

# Load the dataset (Standardized or Non-Standardized)
std = False

if std == True: 
    data = pd.read_csv("STD-PC.csv")
else:
    data = pd.read_csv("CLR-PC.csv")


# Import Data
X = data[['PC1','PC2','PC3','PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9']].values

X_r2 = X

# Grid Search values
cov_types = ["full", "tied", "diag", "spherical"]
n_components_range = range(1, 10)

bic = np.zeros((len(n_components_range), len(cov_types)))

for i, n in enumerate(n_components_range):
    for j, cov in enumerate(cov_types):
        gmm = GaussianMixture(n_components=n, covariance_type=cov, random_state=42)
        gmm.fit(X_r2)
        bic[i, j] = gmm.bic(X_r2)


# Extract the minimum BIC value

min_bic_idx = np.unravel_index(np.argmin(bic), bic.shape)
best_n = n_components_range[min_bic_idx[0]]
best_cov = cov_types[min_bic_idx[1]]

print("Best n_components:", best_n)
print("Best covariance type:", best_cov)



#Plot the BIC curves

plt.figure(figsize=(10, 6))

for j, cov in enumerate(cov_types):
    plt.plot(n_components_range, bic[:, j], label=f'cov={cov}', marker='o')

plt.xlabel("Number of components")
plt.ylabel("BIC")
plt.title("GMM Model Selection via BIC")
plt.legend()
plt.tight_layout()
plt.show()

#What you’re looking for:
# The lowest BIC value = best model.
# The curve shape:
# A gradual decrease → data likely has multiple clusters.
# Sharp dip then rise → clear optimal number.


In [ ]:
# -------------------------
# AIC Search of Parameters
#--------------------------

import pandas as pd
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset (Standardized or Non-Standardized)
std = False

if std == True: 
    data = pd.read_csv("STD-PC.csv")
else:
    data = pd.read_csv("CLR-PC.csv")


# Import Data
X = data[['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9']].values

X_r2 = X

# Grid Search values (Add 'spherical' for first iteration, remove if poor performance)
cov_types = ["full", "tied", "diag", "spherical"]
n_components_range = range(1, 10)

aic = np.zeros((len(n_components_range), len(cov_types)))

for i, n in enumerate(n_components_range):
    for j, cov in enumerate(cov_types):
        gmm = GaussianMixture(
            n_components=n,
            covariance_type=cov,
            random_state=42
        )
        
        gmm.fit(X_r2)
        aic[i, j] = gmm.aic(X_r2)


# Extract the minimum AIC value

min_aic_idx = np.unravel_index(np.argmin(aic), aic.shape)

best_aic = aic[min_aic_idx]
best_n = n_components_range[min_aic_idx[0]]
best_cov = cov_types[min_aic_idx[1]]

print("Best n_components:", best_n)
print("Best covariance type:", best_cov)
print("Best AIC:", best_aic)


# Plot the AIC curves

plt.figure(figsize=(10, 6))

for j, cov in enumerate(cov_types):
    plt.plot(
        n_components_range,
        aic[:, j],
        label=f'cov={cov}',
        marker='o'
    )

plt.xlabel("Number of components")
plt.ylabel("AIC")
plt.title("Till GMM Model Selection via AIC")
plt.legend()
plt.tight_layout()
plt.show()


# What you’re looking for:
#
# The lowest AIC value = best model.


In [ ]:
# -----------------------------------
# Import Bedrock Samples (if needed)
#------------------------------------

# Load bedrock samples PCA scores (CLR or STD)
# Make sure bedrock was fitted to Till PCA
rock = pd.read_csv("BedrockSamples.csv")

rock_X = rock[['PC1','PC2','PC3','PC4', 'PC5', 'PC6', 'PC7']].values  


In [ ]:
# -----------------------
# Gaussian Mixture Model 
#------------------------

import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn import metrics
import matplotlib.pyplot as plt
import itertools
import numpy as np

# Load the dataset (Standardized or Non-Standardized) and 
std = False
save = True

if std == True: 
    data = pd.read_csv("STD-PC.csv")
else:
    data = pd.read_csv("CLR-PC.csv")

# From BIC input best parameters
components = 5
cov = 'diag'

# Apply GaussianMixture
X = data[['PC1','PC2','PC3','PC4', 'PC5', 'PC6', 'PC7']].values


gmm = GaussianMixture(n_components=components, covariance_type=cov, random_state=42).fit(X) 
labels = gmm.predict(X)  # Get cluster labels


# Count occurrences of each cluster label
unique, counts = np.unique(labels, return_counts=True)

# Create a DataFrame without assuming the number of clusters
countshdbclusters = pd.DataFrame({'Cluster': unique, 'Count': counts})
print(countshdbclusters)

# Cluster membership probabilities
probs = gmm.predict_proba(X)   # shape (n_samples, n_components)

# Add results into df
data['GMM_Label'] = labels

# Add each probability column (Cluster_0_prob, Cluster_1_prob, …)
for i in range(probs.shape[1]):
    data[f'GMM_Prob_{i}'] = probs[:, i]

# Preview and save to .csv
print(data.head())
if save == True:
 if std == True:
     data.to_csv(f"GMM-{cov}-{components}-STD.csv", index=False)
 else:
     data.to_csv(f"GMM-{cov}-{components}-CLR.csv", index=False)



In [ ]:
# --------------------------------
# PC1 vs PC2 Biplot with Clusters
# --------------------------------

# Ensure labels are a numpy array
y_num = np.array(labels)

# Define unique cluster labels
unique = np.unique(y_num)

# Dynamically generate cluster names
target_names = {i: f"Cluster {i}" for i in unique if i != -1}

# Get the maxumum probability of assigned clusters
max_probabilities = probs[np.arange(len(labels)), labels]  # Extract the assigned cluster probabilities

# Scale marker sizes non-linearly for better differentiation
min_marker_size = 5  
max_marker_size = 100  

# Transform probabilities to emphasize differences (power scaling)
scaled_sizes = min_marker_size + (max_marker_size - min_marker_size) * ((max_probabilities - min(max_probabilities)) / (max(max_probabilities) - min(max_probabilities)))**2

# Prevent extremely small markers from disappearing
scaled_sizes = np.clip(scaled_sizes, min_marker_size, max_marker_size)

# Plot size settings
plt.figure(figsize=(12, 8))

# Dynamically generate colors and markers
num_clusters = len(unique)
colors = [plt.cm.tab10(i % 10) for i in range(num_clusters)]  # Extract colors from colormap
marker_styles = itertools.cycle(['o', '^', 'x', 's', 'D', 'P', '*', 'v', '<', '>', 'h'])  # Cycle markers

lw = 2

# Get the absolute max of PC1 and PC2 scores
PC1_abs_max = np.abs(X[:, 0]).max()
PC2_abs_max = np.abs(X[:, 1]).max()

# Plot each cluster with varying marker size
for i, color, marker in zip(unique, colors, marker_styles):
    mask = (y_num == i)
    label = target_names.get(i+1, f"Cluster {i+1}")  # Handle dynamic naming
    plt.scatter(X[mask, 0], X[mask, 1], 
            color=color, marker=marker, 
            alpha=0.8, lw=lw, label=label,
            s=scaled_sizes[mask])  # Use improved scaling

# Overlay bedrock samples symbolized by domain on PC1 vs PC2

# Column with domain that sample is from
# Change column to domain or bedrock_type
# bedrock_label_col = "domain"
# bedrock_label = rock[bedrock_label_col].astype(str).values  

# unique_unit = np.unique(bedrock_label)

# # Choose colours for domains
# zone_cmap = plt.cm.tab20  
# zone_colors = {z: zone_cmap(i / max(1, len(unique_unit) - 1)) for i, z in enumerate(unique_unit)}

# Plot rock samples symbolized by bedrock domain
# for z in unique_unit:
#     mask = (bedrock_label == z)
#     plt.scatter(
#         rock_X[mask, 0],   # PC1
#         rock_X[mask, 1],   # PC2
#         marker="*",
#         s=150,
#         edgecolors="black",
#         facecolors=zone_colors[z],
#         linewidths=0.5,
#         label=f"{z}"
#     )


# Plot appearance based on standardized and non-standardized PC variance
# Change %'s manually
if std:
    plt.axvline(0, color='black', linestyle=':')
    plt.axhline(0, color='black', linestyle=':')
    plt.xlabel("PC1 [44.42%]", fontsize=14)             # Manually change
    plt.ylabel("PC2 [30.26%]", fontsize=14)
    plt.legend(loc='best', fontsize=12, title_fontsize=12, frameon=True)
    plt.tight_layout()
else:
    plt.axvline(0, color='black', linestyle=':')
    plt.axhline(0, color='black', linestyle=':')
    plt.xlabel("PC1 [54.74%]", fontsize=14)             # Manually change
    plt.ylabel("PC2 [28.50%]", fontsize=14)   
    plt.legend(loc='best', fontsize=12, title_fontsize=12, frameon=True)
    plt.tight_layout()

# Add loading vectors from PCA Code
if std == True:
    loadings = pd.read_csv("STD-Loading.csv")  # Import standardized loading scores
else:
    loadings = pd.read_csv("CLR-Loading.csv")  # Import non-standardized loading scores

elements = loadings['Elements']

# Scale the loading vectors to fit on the PC plot
scale = 1.5

for i in range(len(elements)):
    # Plot arrows for loadings if needed    
    plt.arrow(0, 0, 
              loadings.loc[i, 'PC1'] * scale, 
              loadings.loc[i, 'PC2'] * scale,
              color='black', alpha=0.7, lw=1.5, head_width=0.08)

    # Add text to arrows (if legible)
    # plt.text(loadings.loc[i, 'PC1'] * scale * 1.1, 
    #          loadings.loc[i, 'PC2'] * scale * 1.1, 
    #          elements[i], color='black', ha='center', va='center', fontsize=15, fontweight='bold')

if save == True:
    if std == True:
        plt.savefig(f'PC1_PC2-{cov}-{components}-STD-Biplot')
    else:
        plt.savefig(f'PC1_PC2-{cov}-{components}-CLR-Biplot_Arrow')
plt.show()

In [ ]:
# --------------------------------
# PC1 vs PC3 Biplot with Clusters
# --------------------------------

# Ensure labels are a numpy array
y_num = np.array(labels)

# Define unique cluster labels
unique = np.unique(y_num)

# Dynamically generate cluster names
target_names = {i: f"Cluster {i}" for i in unique if i != -1}

# Get the probability of assigned clusters
max_probabilities = probs[np.arange(len(labels)), labels]  # Extract the assigned cluster probabilities

# Scale marker sizes non-linearly for better differentiation
min_marker_size = 5  
max_marker_size = 100  

# Transform probabilities to emphasize differences (power scaling)
scaled_sizes = min_marker_size + (max_marker_size - min_marker_size) * ((max_probabilities - min(max_probabilities)) / (max(max_probabilities) - min(max_probabilities)))**2

# Prevent extremely small markers from disappearing
scaled_sizes = np.clip(scaled_sizes, min_marker_size, max_marker_size)

# Plot settings
plt.figure(figsize=(12, 8))

# Dynamically generate colors and markers
num_clusters = len(unique)
colors = [plt.cm.tab10(i % 10) for i in range(num_clusters)]  # Extract colors from colormap
marker_styles = itertools.cycle(['o', '^', 'x', 's', 'D', 'P', '*', 'v', '<', '>', 'h'])  # Cycle markers

lw = 2

# Get the absolute max of PC1 and PC3 scores
PC1_abs_max = np.abs(X[:, 0]).max()
PC3_abs_max = np.abs(X[:, 2]).max()

# Plot each cluster with varying marker size
for i, color, marker in zip(unique, colors, marker_styles):
    mask = (y_num == i)
    label = target_names.get(i+1, f"Cluster {i+1}")  # Handle dynamic naming
    plt.scatter(X[mask, 0], X[mask, 2], 
            color=color, marker=marker, 
            alpha=0.8, lw=lw, label=label,
            s=scaled_sizes[mask])  # Use improved scaling

# Overlay bedrock samples symbolized by domain on PC1 vs PC3

# Column with domain that sample is from
# Change column to domain or bedrock_type
# bedrock_label_col = "domain"
# bedrock_label = rock[bedrock_label_col].astype(str).values  

# unique_unit = np.unique(bedrock_label)

# # Choose colours for domains
# zone_cmap = plt.cm.tab20  
# zone_colors = {z: zone_cmap(i / max(1, len(unique_unit) - 1)) for i, z in enumerate(unique_unit)}

# Plot rock samples symbolized by bedrock domain
# for z in unique_unit:
#     mask = (bedrock_label == z)
#     plt.scatter(
#         rock_X[mask, 0],   # PC1
#         rock_X[mask, 2],   # PC3
#         marker="*",
#         s=150,
#         edgecolors="black",
#         facecolors=zone_colors[z],
#         linewidths=0.5,
#         label=f"{z}"
#     )

# Plot appearance
if std:
    plt.axvline(0, color='black', linestyle=':')
    plt.axhline(0, color='black', linestyle=':')
    plt.xlabel("PC1 [44.42%]", fontsize=14)             # Manually change
    plt.ylabel("PC3 [7.32%]", fontsize=14)   
    plt.legend(loc='best', fontsize=12, title_fontsize=12, frameon=True)
    plt.tight_layout()
else:
    plt.axvline(0, color='black', linestyle=':')
    plt.axhline(0, color='black', linestyle=':')
    plt.xlabel("PC1 [54.74%]", fontsize=14)             # Manually change
    plt.ylabel("PC3 [4.71%]", fontsize=14)  
    plt.legend(loc='best', fontsize=12, title_fontsize=12, frameon=True)
    plt.tight_layout()

# Add loading vectors
if std == True:
    loadings = pd.read_csv("STD-Loading.csv")  # Import standardized loading scores
else:
    loadings = pd.read_csv("CLR-Loading.csv")  # Import non-standardized loading scores

elements = loadings['Elements']

# Scale the vectors to fit nicely on the PC plot
scale = 1.5

for i in range(len(elements)):
    # Plot arrows for loadings if needed    
    plt.arrow(0, 0, 
              loadings.loc[i, 'PC1'] * scale, 
              loadings.loc[i, 'PC3'] * scale,
              color='black', alpha=0.7, lw=1.5, head_width=0.08)

    # Add text to arrows (if legible)
    # plt.text(loadings.loc[i, 'PC1'] * scale , 
    #          loadings.loc[i, 'PC3'] * scale , 
    #          elements[i], color='black', ha='center', va='center', fontsize=15, fontweight='bold')

# Save as .png
if save == True:
    if std == True:
        plt.savefig(f'PC1_PC3-{cov}-{components}-STD-Biplot')
    else:
        plt.savefig(f'PC1_PC3-{cov}-{components}-CLR-Biplot')

plt.show()
